# Notebook 02 — Analisis: dinamica del partido (ht vs ft)

Punto 5 del archivo de ideas. Tres analisis:

1. **Scatter** goles ht vs goles ft por partido, con regresion lineal.
2. **% de remontadas** por equipo (equipo perdia en ht y gano en ft).
3. **Matriz de confusion** ht -> ft (resultado) como heatmap.

> **Estado:** el analisis 1 (scatter) esta implementado. La regresion lineal,
> el analisis 2 y el analisis 3 los completa el companero. Ver `ESTADO_TRABAJO.md`.

## 0. Configuracion y carga del dataset limpio

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / 'src'))

# Cargar el dataset limpio generado en el notebook 01
csv = PROJECT_DIR / 'data' / 'processed' / 'premier_2024_25_limpio.csv'
df = pd.read_csv(csv)
print('Shape:', df.shape)
df.head()

## 1. Scatter goles ht vs ft

¿El medio tiempo predice el tiempo completo? Se cruza el total de goles al medio tiempo (`ht_total`) contra el total al tiempo completo (`ft_total`).

Solo se usan partidos donde ht fue registrado (se excluyen los ~16 con ht null).

In [ ]:
# Filtrar partidos con ht registrado
df_ht = df.dropna(subset=['ht_total','ft_total']).copy()
df_ht['ht_total'] = df_ht['ht_total'].astype(int)
df_ht['ft_total'] = df_ht['ft_total'].astype(int)
print('Partidos con ht:', len(df_ht))
print('Partidos sin ht (excluidos):', len(df) - len(df_ht))

In [ ]:
# Scatter con jitter (los goles son enteros y se superponen)
rng = np.random.default_rng(42)
jitter = 0.12
x = df_ht['ht_total'] + rng.normal(0, jitter, len(df_ht))
y = df_ht['ft_total'] + rng.normal(0, jitter, len(df_ht))

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(x, y, alpha=0.4, edgecolor='none', s=35, color='#1f77b4')
ax.set_xlabel('Goles totales al medio tiempo (ht)')
ax.set_ylabel('Goles totales al tiempo completo (ft)')
ax.set_title('Dinamica del partido: ht vs ft (Premier League 2024/25)')

# Linea de igualdad (ft = ht): si el medio tiempo predijera perfecto
lim = [0, max(df_ht['ft_total'].max(), df_ht['ht_total'].max()) + 1]
ax.plot(lim, lim, '--', color='gray', alpha=0.7, label='ft = ht (sin mas goles)')
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'data' / 'processed' / 'scatter_ht_vs_ft.png', dpi=120)
plt.show()
print('Grafico guardado en data/processed/scatter_ht_vs_ft.png')

In [ ]:
# Correlacion de Pearson entre ht_total y ft_total
from scipy import stats
r, p = stats.pearsonr(df_ht['ht_total'], df_ht['ft_total'])
print(f'Correlacion Pearson r = {r:.3f}  (p = {p:.3e})')
print('Interpretacion: correlacion positiva moderada; el ht aporta senal pero')
print('no determina el ft (hay muchos puntos por encima de la linea de igualdad).')

### 1.1 Regresion lineal  *(pendiente — companero)*

Ajustar `ft_total ~ ht_total` con `statsmodels` o `scipy.stats.linregress`.
Graficar la recta sobre el scatter. Reportar R^2 e interpretar.

```python
# Sugerencia:
# import statsmodels.api as sm
# X = sm.add_constant(df_ht['ht_total'])
# modelo = sm.OLS(df_ht['ft_total'], X).fit()
# print(modelo.summary())
# y_pred = modelo.predict(X)
# ax.plot(df_ht['ht_total'], y_pred, color='red', label='regresion')
```

Tambien se puede repetir el scatter separando goles local vs visitante
(ht_g1 vs ft_g1 y ht_g2 vs ft_g2) para ver si la relacion difiere.

## 2. % de remontadas por equipo  *(pendiente — companero)*

Una **remontada** es un partido donde el equipo perdia en ht y gano en ft.
La columna `remontada` ya esta calculada en el dataset limpio.

Pasos sugeridos:
1. Para cada equipo, contar partidos donde fue local o visitante y tenia ht.
2. Contar cuantas veces remonto (perdia en ht, gano en ft).
3. Calcular % = remontadas / partidos_con_ht * 100.
4. Graficar bar chart horizontal de los equipos con mayor %.

> Nota: hay que decidir si se cuenta la remontada del local o del visitante
> por separado. Una opcion es calcular remontadas 'a favor' del equipo
> (fuera local o visitante).

## 3. Matriz de confusion ht -> ft (heatmap)  *(pendiente — companero)*

Cruzar `res_ht` (L/E/V) contra `res_ft` (L/E/V) en una tabla 3x3 de conteos
o porcentajes. Visualizar como heatmap con `sns.heatmap`.

```python
# Sugerencia:
# tabla = pd.crosstab(df_ht['res_ht'], df_ht['res_ft'], normalize='index')
# sns.heatmap(tabla, annot=True, cmap='Blues')
```

La diagonal son los partidos donde el resultado no cambio entre ht y ft.
Las celdas fuera de la diagonal son los cambios de resultado (incluye remontadas).

## Resumen de lo implementado vs pendiente

| Analisis | Estado | Responsable |
|---|---|---|
| Scatter ht vs ft | Hecho | [yo] |
| Correlacion Pearson | Hecho | [yo] |
| Regresion lineal sobre scatter | Pendiente | companero |
| % remontadas por equipo | Pendiente | companero |
| Matriz de confusion ht->ft | Pendiente | companero |